# Сравнение градиентных бустингов и Optuna
Продолжение `random_forest_model.ipynb`. Требуется файл `rent_apartments.csv` рядом с ноутбуком. Сравнение проводится **только на обучающей выборке** через одинаковую 5-fold CV; тестовая выборка используется один раз в конце.

## 0. Зависимости
При необходимости выполните следующую команду в терминале: `pip install pandas numpy scikit-learn xgboost lightgbm catboost optuna joblib matplotlib`

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    KFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from xgboost import XGBRegressor

RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS = 40  # увеличьте до 100–200 для более тщательного поиска


## 1. Подготовка данных
Сохраняем исходный набор признаков. Исправляем преобразование `garden`, избегая присваивания по цепочке.

In [ ]:
data = pd.read_csv('rent_apartments.csv')
feature_cols = ['area', 'constraction_year', 'bedrooms', 'garden',
                'balcony_yes', 'parking_yes', 'furnished_yes', 'garage_yes', 'storage_yes']
encoded = pd.get_dummies(data.copy(), columns=['balcony', 'parking', 'furnished', 'garage', 'storage'], drop_first=True)
encoded['garden'] = (encoded['garden'].astype(str).str.extract(r'(\d+)', expand=False)
                     .fillna('0').astype(float))
for col in feature_cols:
    if col not in encoded.columns:
        encoded[col] = 0
X = encoded[feature_cols].apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(encoded['rent'], errors='coerce')
valid = y.notna()
X, y = X.loc[valid], y.loc[valid]
# Для деревьев заполняем пропуски медианой, рассчитанной ТОЛЬКО на train.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
train_medians = X_train.median().fillna(0)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 2. Сравнение пяти моделей
Главный критерий — **минимальный CV RMSE**. MAE и R² показываем дополнительно. Это предварительное сравнение моделей с базовыми настройками.

In [ ]:
models = {
    'Random Forest': RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
    'XGBoost': XGBRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=1, objective='reg:squarederror'),
    'LightGBM': LGBMRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=1, verbosity=-1),
    'CatBoost': CatBoostRegressor(iterations=300, random_seed=RANDOM_STATE, verbose=False, thread_count=1),
}
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = {'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'}
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    rows.append({'model': name, 'CV_MAE': -scores['test_mae'].mean(),
                 'CV_RMSE': -scores['test_rmse'].mean(), 'CV_R2': scores['test_r2'].mean(),
                 'CV_RMSE_std': scores['test_rmse'].std()})
results = pd.DataFrame(rows).sort_values('CV_RMSE').reset_index(drop=True)
display(results)
winner_name = results.loc[0, 'model']
print('Лучшая базовая модель:', winner_name)


## 3. Автоматический подбор гиперпараметров Optuna
Настраиваем победителя первого этапа, минимизируя средний RMSE на тех же CV-фолдах. Для полностью несмещённой оценки процесса выбора модели можно использовать nested CV; здесь финальный holdout-test остаётся нетронутым.

In [ ]:
def suggest_model(trial, name):
    if name == 'Random Forest':
        return RandomForestRegressor(
            n_estimators=trial.suggest_int('n_estimators', 100, 800, step=100),
            max_depth=trial.suggest_int('max_depth', 3, 25),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
            max_features=trial.suggest_float('max_features', 0.5, 1.0),
            random_state=RANDOM_STATE, n_jobs=1)
    if name == 'Gradient Boosting':
        return GradientBoostingRegressor(
            n_estimators=trial.suggest_int('n_estimators', 100, 700, step=100),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            max_depth=trial.suggest_int('max_depth', 2, 7),
            subsample=trial.suggest_float('subsample', 0.6, 1.0),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
            random_state=RANDOM_STATE)
    if name == 'XGBoost':
        return XGBRegressor(
            n_estimators=trial.suggest_int('n_estimators', 100, 800, step=100),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            max_depth=trial.suggest_int('max_depth', 2, 10),
            subsample=trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 20, log=True),
            objective='reg:squarederror', random_state=RANDOM_STATE, n_jobs=1)
    if name == 'LightGBM':
        return LGBMRegressor(
            n_estimators=trial.suggest_int('n_estimators', 100, 800, step=100),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int('num_leaves', 7, 63),
            min_child_samples=trial.suggest_int('min_child_samples', 5, 50),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 20, log=True),
            random_state=RANDOM_STATE, n_jobs=1, verbosity=-1)
    if name == 'CatBoost':
        return CatBoostRegressor(
            iterations=trial.suggest_int('iterations', 100, 800, step=100),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            depth=trial.suggest_int('depth', 3, 9),
            l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 20, log=True),
            random_seed=RANDOM_STATE, verbose=False, thread_count=1)
    raise ValueError(name)

def objective(trial):
    model = suggest_model(trial, winner_name)
    scores = cross_val_score(model, X_train, y_train, cv=cv,
                             scoring='neg_root_mean_squared_error', n_jobs=1)
    return -scores.mean()

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS)
print('Победитель:', winner_name)
print('Оптимальный CV RMSE:', study.best_value)
print('Лучшие параметры:', study.best_params)


## 4. Финальное обучение и оценка на test

In [ ]:
best_model = suggest_model(optuna.trial.FixedTrial(study.best_params), winner_name)
best_model.fit(X_train, y_train)
pred = best_model.predict(X_test)
metrics = {'Test_MAE': mean_absolute_error(y_test, pred),
           'Test_RMSE': np.sqrt(mean_squared_error(y_test, pred)),
           'Test_R2': r2_score(y_test, pred)}
print(pd.Series(metrics))


## 5. Сохранение модели и схемы подготовки признаков

In [ ]:
Path('models').mkdir(exist_ok=True)
artifact = {'model': best_model, 'model_name': winner_name,
            'features': feature_cols, 'train_medians': train_medians,
            'best_params': study.best_params, 'test_metrics': metrics}
joblib.dump(artifact, 'models/best_boosting_model.joblib')
results.to_csv('boosting_comparison.csv', index=False)
print('Сохранены models/best_boosting_model.joblib и boosting_comparison.csv')


### Важно
- Без `rent_apartments.csv` получить реальные значения метрик невозможно.
- Если цель — сравнить **каждый бустинг после настройки**, запускайте отдельное Optuna-исследование для каждого алгоритма и сравнивайте лучшие CV RMSE, не используя test для выбора.
- В исходном ноутбуке `garden` изменяется поэлементно; здесь этот шаг сделан векторно.
- Если категориальные поля содержат неожиданные значения, проверьте результат `get_dummies` и схему признаков.